# Direction Evaluator — Calibration (STEP 10A–10F)

Owner: member A — this notebook belongs to one person, so editing it never conflicts with others.

The Direction calibration workflow from Df5, unchanged. The evaluator code is in `evaluation/eval_trajectory.py`; the current rule used by `run_benchmark.ipynb` is `TrajectoryRequirementEvaluator.CURRENT_MIN_DISPLACEMENT` in that file.

Human Gold Labels are read from `labels/<MODEL_NAME>_pilot_human_gold_labels.json` (create them in STEP 9D/9E of `run_benchmark.ipynb`).

## STEP 0 — Get the code from GitHub

Clones the repository (first run) or pulls the latest version, then imports `evaluation/common.py` and **every `evaluation/eval_*.py` automatically** — a new evaluator file is picked up without editing this notebook.

- `BRANCH = "main"` for normal use; set it to your branch name to test your work before it is merged.
- Private repository only: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon, left sidebar).
- CPU runtime is enough for evaluation (no GPU needed).

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Soniaaaa-aa/t2m-capability-benchmark"
BRANCH = "main"                                   # or your feature branch
REPO_DIR = Path("/content/t2m-capability-benchmark")

def _git(*args, cwd=None):
    print("$ git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True)

url = REPO_URL
try:  # optional token for a private repository
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if not (REPO_DIR / ".git").exists():
    _git("clone", "--branch", BRANCH, url, str(REPO_DIR))
else:
    _git("fetch", "origin", cwd=REPO_DIR)
    _git("checkout", BRANCH, cwd=REPO_DIR)
    _git("pull", "origin", BRANCH, cwd=REPO_DIR)

EVAL_DIR = REPO_DIR / "evaluation"
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

import common
importlib.reload(common)                 # pick up changes after a pull
from common import *                     # settings, loaders, registry, gold helpers, runner
evaluator_modules = load_all_evaluators(EVAL_DIR)   # imports every eval_*.py

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("Framework commit     :", commit or "unknown")

## STEP 1 — Load the model's motions, cases and Human Gold Labels

One call replaces STEP 3.5–9F of `run_benchmark.ipynb` (upload prompt appears only if the ZIP / benchmark JSON is missing).

In [ ]:
MODEL_NAME = "MoMADiff"      # ← model to analyse

data = load_inputs(MODEL_NAME, REPO_DIR)
evaluation_cases = data["evaluation_cases"]
HUMAN_GOLD_LABELS = data["human_gold_labels"]

In [ ]:
from eval_trajectory import TrajectoryEvaluator, calculate_required_direction_ratio, DirectionDecisionRule

if HUMAN_GOLD_LABELS is None:
    raise RuntimeError("Human Gold Labels are required for calibration — create them in run_benchmark.ipynb STEP 9D/9E.")

## STEP 10A — TrajectoryEvaluator

This step defines the evaluator used to collect **Trajectory Evidence** for Direction Requirements (`forward`, `backward`, `left`, and `right`).

### What this evaluator measures

The evaluator extracts the trajectory of the root joint (Joint 0) from the Standardised Motion `[T, 22, 3]` and analyses movement on the XZ plane.

The benchmark coordinate system is:

- `+X` = Right
- `-X` = Left
- `+Z` = Forward
- `-Z` = Backward
- `+Y` = Up

For each motion, the evaluator calculates:

- `dx` — displacement along the X-axis
- `dz` — displacement along the Z-axis
- `total_displacement` — straight-line displacement on the XZ plane
- `dominant_axis` — whether X or Z has the larger displacement
- `dominance_ratio` — relative dominance of the larger axis
- `raw_direction` — preliminary direction estimated from the displacement

### Important

This step **does not yet make the final PASS / FAIL decision**. The evaluator first collects measurable Evidence. The final Direction rule and thresholds are calibrated later using the Pilot Human Gold Labels.

**Input:** Standardised Motion `[T, 22, 3]`  
**Output:** Direction Trajectory Evidence

### STEP 10A-1 — Quick Sanity Check

Before running calibration on all Pilot cases, test the `TrajectoryEvaluator` on one registered motion with a `direction = left` Requirement.

This verifies that:

- the `.npy` motion can be loaded correctly,
- the root trajectory can be extracted,
- `dx`, `dz`, dominant axis, dominance ratio, and raw direction are produced,
- PASS / FAIL remains unset before threshold calibration.

This cell is only a **sanity check** and does not determine the final Direction rule.

In [ ]:
# Find an Evaluation Case containing a direction=left Requirement
left_case = None
for case in evaluation_cases:
    for req in case["requirements"]:
        value = req.get("value", req.get("expected"))
        if req.get("type") == "direction" and str(value).lower() == "left":
            left_case = case
            break
    if left_case is not None:
        break

if left_case is None:
    raise RuntimeError("No Evaluation Case with direction=left was found.")

motion = np.load(left_case["motion_path"], allow_pickle=False)
motion = validate_standardised_motion(motion)

evaluator = TrajectoryEvaluator()
result = evaluator.evaluate(motion=motion, required_direction="left")
evidence = result["evidence"]

print("Motion used:", left_case["motion_path"])
print("Shape      :", motion.shape)
print("Required Direction:", result["required_direction"])
print("Predicted Direction:", result["predicted_direction_raw"])
print(f"dx: {evidence['dx']:.4f} m")
print(f"dz: {evidence['dz']:.4f} m")
print("Dominant axis :", evidence["dominant_axis"])
print("Dominance ratio:", evidence["dominance_ratio"])
print("PASS / FAIL   :", result["pass_fail"], "(Threshold not set)")

## STEP 10B — Collect Direction Evidence and Link Human Gold

Apply `TrajectoryEvaluator` to every Pilot case containing a `direction` Requirement and link each Requirement to the Human Gold Label created in STEP 9D.

For each Direction Requirement, this step records:

- required direction,
- Human Gold Label,
- raw predicted direction,
- X and Z displacement,
- displacement in the required direction,
- orthogonal displacement,
- required-direction ratio.

The purpose is to create the calibration dataset used to design the Direction decision rule.

`UNCERTAIN` labels are retained here for inspection, but they should not be treated as PASS or FAIL when fitting a threshold.

In [ ]:
# get_human_label (common.py) and calculate_required_direction_ratio (eval_trajectory.py) are imported

# ============================================================
# 3. Direction Requirementを持つEvaluation Caseを評価
# ============================================================

direction_results = []

evaluator = TrajectoryEvaluator()


for case in evaluation_cases:

    # --------------------------------------------------------
    # RequirementをIndex付きで取得
    # --------------------------------------------------------

    for req_index, req in enumerate(
        case["requirements"]
    ):

        # Direction Requirement以外は今回は無視
        if req.get("type") != "direction":
            continue


        # ----------------------------------------------------
        # Required Direction
        # ----------------------------------------------------

        required_direction = req.get(
            "value",
            req.get("expected")
        )

        if required_direction is None:
            continue

        required_direction = str(
            required_direction
        ).lower()


        # ----------------------------------------------------
        # Prompt ID
        # ----------------------------------------------------

        prompt_id = (
            case.get("prompt_id")
            or case.get("id")
            or case.get("case_id")
            or "UNKNOWN"
        )


        # ----------------------------------------------------
        # Motion読み込み
        # ----------------------------------------------------

        motion = np.load(
            case["motion_path"],
            allow_pickle=False
        )

        motion = validate_standardised_motion(
            motion
        )


        # ----------------------------------------------------
        # TrajectoryEvaluator
        # ----------------------------------------------------

        result = evaluator.evaluate(
            motion=motion,
            required_direction=required_direction
        )

        evidence = result["evidence"]

        dx = evidence["dx"]
        dz = evidence["dz"]


        # ----------------------------------------------------
        # Required-direction Evidence
        # ----------------------------------------------------

        (
            required_displacement,
            orthogonal_displacement,
            required_ratio,
        ) = calculate_required_direction_ratio(

            dx=dx,

            dz=dz,

            required_direction=required_direction,
        )


        # ----------------------------------------------------
        # Human Gold Label
        #
        # STEP 9DのHUMAN_GOLD_LABELSから取得
        # ----------------------------------------------------

        human_label = get_human_label(
            HUMAN_GOLD_LABELS,
            case=case,
            requirement_index=req_index,
        )


        # ----------------------------------------------------
        # 結果を保存
        # ----------------------------------------------------

        direction_results.append({

            "prompt_id":
                prompt_id,

            "requirement_index":
                req_index,

            "motion_path":
                case["motion_path"],

            "required_direction":
                required_direction,

            "human_label":
                human_label,

            "predicted_raw":
                result["predicted_direction_raw"],

            "dx":
                dx,

            "dz":
                dz,

            "required_displacement":
                required_displacement,

            "orthogonal_displacement":
                orthogonal_displacement,

            "required_ratio":
                required_ratio,

            "dominant_axis":
                evidence["dominant_axis"],

            "dominance_ratio":
                evidence["dominance_ratio"],
        })


# ============================================================
# 4. Display results
# ============================================================

print(
    f"Direction Requirement Cases: "
    f"{len(direction_results)}"
)

print("=" * 100)


for i, r in enumerate(
    direction_results,
    start=1
):

    print(f"\n[{i}]")

    print(
        "Prompt ID     :",
        r["prompt_id"]
    )

    print(
        "Req Index     :",
        r["requirement_index"]
    )

    print(
        "Motion        :",
        r["motion_path"]
    )

    print(
        "Required      :",
        r["required_direction"]
    )

    print(
        "Human Label   :",
        r["human_label"]
    )

    print(
        "Predicted Raw :",
        r["predicted_raw"]
    )

    print(
        f"dx            : "
        f"{r['dx']:.4f} m"
    )

    print(
        f"dz            : "
        f"{r['dz']:.4f} m"
    )

    print(
        f"Required Disp : "
        f"{r['required_displacement']:.4f} m"
    )

    print(
        f"Orthogonal    : "
        f"{r['orthogonal_displacement']:.4f} m"
    )

    print(
        f"Required Ratio: "
        f"{r['required_ratio']:.4f}"
    )

    print("-" * 100)


# ============================================================
# 5. Human Label確認
# ============================================================

print("\n" + "=" * 100)
print("HUMAN LABEL SUMMARY")
print("=" * 100)

pass_count = sum(
    r["human_label"] == "PASS"
    for r in direction_results
)

fail_count = sum(
    r["human_label"] == "FAIL"
    for r in direction_results
)

uncertain_count = sum(
    r["human_label"] == "UNCERTAIN"
    for r in direction_results
)

none_count = sum(
    r["human_label"] is None
    for r in direction_results
)


print("PASS      :", pass_count)
print("FAIL      :", fail_count)
print("UNCERTAIN :", uncertain_count)
print("None      :", none_count)


if none_count == 0:
    print("\nHuman Gold Labels linked successfully ✅")
else:
    print(
        "\nWARNING: Human Labelが取得できていない"
        "Requirementがあります。"
    )


print("=" * 100)
print("STEP 10B — Direction Evidence collected ✅")
print("=" * 100)

## STEP 10C — Candidate Minimum-Displacement Threshold Search

Use the Pilot Direction cases from STEP 10B to compare candidate **minimum required-displacement thresholds**.

For calibration:

- Human `PASS` and `FAIL` cases are used.
- Human `UNCERTAIN` cases are excluded from threshold fitting.
- Each candidate threshold produces an automatic PASS / FAIL prediction.
- The prediction is compared with Human Gold using Accuracy, False PASS, and False FAIL.

### Important

The best value found here is a **Pilot candidate threshold**, not yet the final benchmark threshold.

The Direction rule can be refined using Pilot evidence and cross-model validation. Once the evaluator design is finalised, the rule should be **frozen before evaluation of the remaining benchmark prompts**.

In [ ]:
# ============================================================
# STEP 10C — Candidate Direction Threshold Search
# Compare Minimum Displacement candidates using Pilot Human Gold
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. Candidate Thresholds used for calibration
# ------------------------------------------------------------

candidate_thresholds = [
    0.10,
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
]


# ------------------------------------------------------------
# 2. Exclude UNCERTAIN
# ------------------------------------------------------------

calibration_cases = [
    r for r in direction_results
    if r["human_label"] in {"PASS", "FAIL"}
]


print("=" * 100)
print("DIRECTION THRESHOLD CALIBRATION")
print("=" * 100)

print(
    "Usable Human-labelled cases:",
    len(calibration_cases)
)

print(
    "UNCERTAIN excluded:",
    len(direction_results) - len(calibration_cases)
)


# ------------------------------------------------------------
# 3. Evaluate each Candidate Threshold
# ------------------------------------------------------------

threshold_results = []


for threshold in candidate_thresholds:

    correct = 0
    total = 0

    false_pass = 0
    false_fail = 0

    case_results = []


    for r in calibration_cases:

        required_disp = r[
            "required_displacement"
        ]

        human_label = r[
            "human_label"
        ]


        # ----------------------------------------------------
        # Candidate Direction Rule
        #
        # 1. Movement in the required direction is correct
        # 2. Required displacement >= threshold
        # ----------------------------------------------------

        if required_disp >= threshold:

            predicted_label = "PASS"

        else:

            predicted_label = "FAIL"


        # ----------------------------------------------------
        # Compare with Human Gold
        # ----------------------------------------------------

        is_correct = (
            predicted_label == human_label
        )

        if is_correct:
            correct += 1

        elif (
            predicted_label == "PASS"
            and human_label == "FAIL"
        ):
            false_pass += 1

        elif (
            predicted_label == "FAIL"
            and human_label == "PASS"
        ):
            false_fail += 1


        total += 1


        case_results.append({

            "prompt_id":
                r["prompt_id"],

            "human_label":
                human_label,

            "predicted_label":
                predicted_label,

            "required_displacement":
                required_disp,

            "correct":
                is_correct,
        })


    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )


    threshold_results.append({

        "threshold":
            threshold,

        "correct":
            correct,

        "total":
            total,

        "accuracy":
            accuracy,

        "false_pass":
            false_pass,

        "false_fail":
            false_fail,

        "cases":
            case_results,
    })


# ------------------------------------------------------------
# 4. Compare thresholds
# ------------------------------------------------------------

print("\n")
print(
    f"{'Threshold':<12}"
    f"{'Correct':<12}"
    f"{'Accuracy':<12}"
    f"{'False PASS':<14}"
    f"{'False FAIL':<14}"
)

print("-" * 70)


for result in threshold_results:

    print(
        f"{result['threshold']:<12.2f}"
        f"{result['correct']:<12}"
        f"{result['accuracy']:<12.3f}"
        f"{result['false_pass']:<14}"
        f"{result['false_fail']:<14}"
    )


# ------------------------------------------------------------
# 5. Check best accuracy
# ------------------------------------------------------------

best_accuracy = max(
    r["accuracy"]
    for r in threshold_results
)

best_candidates = [
    r for r in threshold_results
    if r["accuracy"] == best_accuracy
]


print("\n" + "=" * 100)
print("BEST CANDIDATE(S)")
print("=" * 100)


for result in best_candidates:

    print(
        f"Threshold = "
        f"{result['threshold']:.2f} m"
        f" | Accuracy = "
        f"{result['accuracy']:.3f}"
    )


print("=" * 100)
print(
    "NOTE: These are PILOT candidate thresholds, "
    "not final benchmark thresholds."
)
print("=" * 100)

## STEP 10D — Define the Initial Direction Rule

Based on the Pilot calibration results from STEP 10C, define the **Initial Direction Decision Rule**.

### Initial Pilot Rule

A Direction requirement is predicted as PASS only when both conditions are satisfied:

1. The motion has the correct sign for the required direction.
2. The displacement in the required direction is at least the Minimum Displacement Threshold.

Direction sign conditions:

- `forward` → `dz > 0`
- `backward` → `dz < 0`
- `right` → `dx > 0`
- `left` → `dx < 0`

Because a Human-PASS Pilot motion can move diagonally, `dominant_axis` is **not** used as a mandatory PASS condition. `dominant_axis` and `dominance_ratio` are retained as diagnostic evidence.

### Current Status

**Pilot Candidate — NOT FROZEN**

Current candidate threshold: **0.50 m**

The same initial rule must first be applied to all Pilot models. **Do not tune the threshold separately for each model.**

After Cross-model Validation, the rule and/or threshold can be refined if the evidence shows that a change is necessary.

`DirectionDecisionRule` is defined in `evaluation/eval_trajectory.py`. When the rule or threshold changes, update `TrajectoryRequirementEvaluator.CURRENT_MIN_DISPLACEMENT` in that file (via a Pull Request) so `run_benchmark.ipynb` uses it.

## STEP 10E — Validate the Initial Direction Rule on the Current Model Pilot

Apply the Initial Direction Rule from STEP 10D to the Pilot Direction cases for the model selected in STEP 4.

Compare the Automatic Prediction with that model's Human Gold Labels and inspect:

- Accuracy
- False PASS
- False FAIL
- Mismatch cases

Human Gold cases labelled `UNCERTAIN` are excluded from the accuracy calculation.

This is **Current Model Pilot Validation**, not the final rule confirmation.

In [ ]:
# ============================================================
# STEP 10E — Current Model Pilot Validation
# ============================================================

direction_rule = DirectionDecisionRule(
    min_displacement=0.50
)

validation_results = []

print("=" * 110)
print(f"INITIAL DIRECTION RULE — PILOT VALIDATION [{MODEL_NAME}]")
print("=" * 110)

for result in direction_results:

    evidence = {
        "dx": result["dx"],
        "dz": result["dz"],
        "dominant_axis": result["dominant_axis"],
        "dominance_ratio": result["dominance_ratio"],
    }

    auto_result = direction_rule.evaluate(
        evidence=evidence,
        required_direction=result["required_direction"],
    )

    human_label = result["human_label"]
    prediction = auto_result["prediction"]

    if human_label == "UNCERTAIN":
        match = None
        status = "EXCLUDED"
    else:
        match = prediction == human_label
        status = "MATCH" if match else "MISMATCH"

    validation_results.append({
        "model": MODEL_NAME,
        "prompt_id": result["prompt_id"],
        "requirement_index": result["requirement_index"],
        "required_direction": result["required_direction"],
        "human_label": human_label,
        "prediction": prediction,
        "dx": result["dx"],
        "dz": result["dz"],
        "required_displacement": auto_result["required_displacement"],
        "correct_sign": auto_result["correct_sign"],
        "sufficient_displacement": auto_result["sufficient_displacement"],
        "match": match,
        "status": status,
    })

usable = [r for r in validation_results if r["match"] is not None]
correct = sum(1 for r in usable if r["match"])
accuracy = correct / len(usable) if usable else 0.0

false_pass = sum(
    1 for r in usable
    if r["human_label"] == "FAIL" and r["prediction"] == "PASS"
)

false_fail = sum(
    1 for r in usable
    if r["human_label"] == "PASS" and r["prediction"] == "FAIL"
)

print(f"Usable Cases       : {len(usable)}")
print(f"Correct Predictions: {correct}")
print(f"Accuracy           : {accuracy:.3f}")
print(f"False PASS         : {false_pass}")
print(f"False FAIL         : {false_fail}")
print(f"UNCERTAIN Excluded : {len(validation_results) - len(usable)}")

mismatches = [r for r in usable if not r["match"]]

if mismatches:
    print("\nMISMATCH CASES")
    print("-" * 110)
    for r in mismatches:
        print(
            r["prompt_id"],
            "| Required:", r["required_direction"],
            "| Human:", r["human_label"],
            "| Auto:", r["prediction"],
            "| Required displacement:",
            f"{r['required_displacement']:.4f} m",
        )
else:
    print("\nNo PASS/FAIL mismatches found for this model.")

## STEP 10F — Cross-model Direction Validation

Now apply the same notebook and the same Initial Direction Rule to the other models.

For each model, the team member should:

1. Change `MODEL_NAME` in STEP 4 to their model name.
2. Prepare `.npy` files in the common Standardised Motion format.
3. Create or load Human Gold Labels for that model's Pilot motions.
4. Run STEP 10A–10E using the **same Direction Rule and the same 0.50 m candidate threshold**.
5. Record all mismatch cases and their evidence.

### Purpose of Cross-model Validation

The goal is to determine whether the Initial Rule developed from the first Pilot model also agrees with Human Gold on other models.

If mismatches occur, inspect the evidence before changing the rule. Determine whether:

- the threshold is inappropriate,
- an additional condition is required, or
- a complex motion needs event/segment-level evaluation.

**Do not change the threshold simply because one mismatch appears.**

After results from the Pilot models have been compared, refine the rule only if supported by the cross-model evidence. If the rule is changed, re-test the updated rule across the Pilot models.

Once the evaluator design is finalised, **freeze the Final Direction Rule**.

After Rule Freeze, apply exactly the same rule to the remaining Benchmark Prompts without further tuning.

### Important Design Principle

Human Gold Labels are created separately for each model, because each model generates different motions.

However, the final Automatic Evaluator Rule must be **shared across all models**.

This ensures that different T2M models are compared using the same evaluation criteria.